# 進捗バーを表示する (tqdm)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/01_colab_tips/tqdm_progress.ipynb)

NLP クックブック 01-08 ／ Google Colab の小技

時間のかかる処理が、いまどこまで進んでいて、あと何分かかるのかを知りたい。

In [1]:
import time
from tqdm.auto import tqdm   # ノートブックでもスクリプトでも適切な表示になる

texts = [f"文 {i}" for i in range(200)]
results = []
for text in tqdm(texts, desc="処理中"):
    time.sleep(0.005)
    results.append(len(text))

In [2]:
# バッチ処理: 件数ベースで進める / 途中経過 (損失など) を右側に表示する
batch_size = 32
with tqdm(total=len(texts), desc="推論", unit="文") as pbar:
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        time.sleep(0.02)
        pbar.update(len(batch))
        pbar.set_postfix(loss=f"{1 / (i + 1):.4f}")

In [3]:
# pandas の apply に進捗バーを付ける
import pandas as pd

tqdm.pandas(desc="apply")
df = pd.DataFrame({"text": texts})
df["length"] = df["text"].progress_apply(len)

# ジェネレータなど長さの分からないものは total を渡す。並列処理は tqdm.contrib.concurrent が便利
from tqdm.contrib.concurrent import thread_map

lengths = thread_map(lambda t: (time.sleep(0.005), len(t))[1], texts, max_workers=8, desc="並列")
print(sum(lengths))

890


- `tqdm.auto` を使うと、Colab ではウィジェットの進捗バー、端末ではテキストの進捗バーになる。
- ループの中で `print` するとバーが崩れる。代わりに `tqdm.write("...")` を使う。
- API 呼び出しのような待ち時間の長い処理は、`thread_map` でスレッド並列にすると大幅に速くなる (レート制限に注意)。